<a href="https://colab.research.google.com/github/kkorotkovuky/bch612-colab/blob/main/MD_designed_peptide_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Molecular dynamics of a designed peptide (BCH612, 2026)
Part 2 of the design exercise. Take the structure you designed with the AlphaFold2 single-sequence notebook, put it in a box of water with salt, and simulate it with [OpenMM](https://openmm.org) using an Amber force field. Then ask: **does the design stay folded?**

The notebook keeps the logic of the Making-it-rain Amber notebook (Arantes *et al.*), in fewer steps, and installs with `pip` only, so it runs on the current Colab runtime (Python 3.13).

### Steps
1. **Runtime → Change runtime type → T4 GPU.**
2. Run the cells in order. Each cell has a ▶ button.
3. When asked, upload the `.pdb` file you downloaded from the AlphaFold notebook.

A 1 ns simulation of a 30–40-residue peptide takes a few minutes on a T4 GPU.

In [ ]:
#@title 1. Install OpenMM, PDBFixer and MDTraj (about 1 min)
import subprocess, sys, importlib.util
PKGS = "openmm[cuda12]==8.6.1 pdbfixer==1.12.0 mdtraj==1.11.2 py3Dmol"
if any(importlib.util.find_spec(m) is None for m in ["openmm", "pdbfixer", "mdtraj", "py3Dmol"]):
  print("Installing ...")
  r = subprocess.run(f"{sys.executable} -m pip -q install {PKGS}", shell=True, capture_output=True, text=True)
  if r.returncode != 0:
    raise RuntimeError("pip install failed:\n" + r.stderr[-2000:])

import openmm

def _platform_works(name):
  # a platform can be listed but still unusable (e.g. no GPU attached), so try it
  try:
    s = openmm.System(); s.addParticle(1.0)
    openmm.Context(s, openmm.VerletIntegrator(0.001), openmm.Platform.getPlatformByName(name))
    return True
  except Exception:
    return False

PLATFORM = next(p for p in ["CUDA", "OpenCL", "CPU"] if _platform_works(p))
print("OpenMM", openmm.__version__, "| using platform:", PLATFORM)
if PLATFORM != "CUDA":
  print("WARNING: no GPU found, the simulation will be much slower.",
        "Use Runtime -> Change runtime type -> T4 GPU, then run this cell again.")

In [ ]:
#@title 2. Upload your designed structure (PDB file from Part 1)
#@markdown Leave `pdb_file` empty to get an upload button.
pdb_file = "" #@param {type:"string"}
import os
import numpy as np
if pdb_file == "" or not os.path.isfile(pdb_file):
  from google.colab import files
  uploaded = files.upload()
  pdb_file = list(uploaded.keys())[0]

# AlphaFold writes pLDDT into the B-factor column; keep it to compare with the simulation
PLDDT = np.array([float(l[60:66]) for l in open(pdb_file)
                  if l.startswith("ATOM") and l[12:16].strip() == "CA"])
import mdtraj as md
_t = md.load(pdb_file)
SEQUENCE = "".join(r.code for r in _t.topology.residues if r.is_protein)
print(f"{pdb_file}: {len(SEQUENCE)} residues, {_t.topology.n_chains} chain(s)")
print("sequence:", SEQUENCE)
print(f"AlphaFold mean pLDDT: {PLDDT.mean():.1f}")

In [ ]:
#@title 3. Build the system: add hydrogens, water and salt
Force_field = "ff14SB + TIP3P water" #@param ["ff14SB + TIP3P water", "ff19SB + OPC water"]
Box_padding_nm = 1.0 #@param {type:"slider", min:0.8, max:1.5, step:0.1}
NaCl_molar = 0.15 #@param {type:"number"}

from openmm import app, unit
from pdbfixer import PDBFixer

if Force_field.startswith("ff14SB"):
  FF_FILES, WATER = ("amber14-all.xml", "amber14/tip3p.xml"), "tip3p"
else:
  FF_FILES, WATER = ("amber19-all.xml", "amber19/opc.xml"), "tip4pew"  # OPC is a 4-site water; placed with the TIP4P-Ew geometry
forcefield = app.ForceField(*FF_FILES)

# complete the termini and add hydrogens at pH 7
fixer = PDBFixer(filename=pdb_file)
fixer.findMissingResidues(); fixer.missingResidues = {}
fixer.findMissingAtoms(); fixer.addMissingAtoms()
fixer.addMissingHydrogens(7.0)
with open("design_prepared.pdb", "w") as f:
  app.PDBFile.writeFile(fixer.topology, fixer.positions, f)

modeller = app.Modeller(fixer.topology, fixer.positions)
modeller.addSolvent(forcefield, model=WATER, padding=Box_padding_nm * unit.nanometer,
                    boxShape="dodecahedron",   # ~29% less water than a cube with the same padding
                    ionicStrength=NaCl_molar * unit.molar, neutralize=True)
TOPOLOGY, POSITIONS = modeller.topology, modeller.positions

# hydrogen mass repartitioning allows a 4 fs time step
system = forcefield.createSystem(TOPOLOGY, nonbondedMethod=app.PME,
                                 nonbondedCutoff=1.0 * unit.nanometer,
                                 constraints=app.HBonds, hydrogenMass=1.5 * unit.amu)
# disulfides: OpenMM bonds SG atoms closer than 3 A when it builds the topology,
# so they form from the geometry alone, with or without SSBOND records in the file
SS_PAIRS = [(b[0].index, b[1].index, b[0].residue.id, b[1].residue.id)
            for b in TOPOLOGY.bonds() if b[0].name == "SG" and b[1].name == "SG"]
n_free_sh = sum(1 for r in TOPOLOGY.residues() if r.name == "CYS"
                and any(a.name == "HG" for a in r.atoms()))
def ss_distances(positions):
  p = np.array([[v.x, v.y, v.z] for v in positions]) * 10.0
  return [(f"{r1}-{r2}", float(np.linalg.norm(p[i] - p[j]))) for i, j, r1, r2 in SS_PAIRS]
if SS_PAIRS:
  print(f"{len(SS_PAIRS)} disulfide bond(s) formed: " +
        ", ".join(f"{t} ({d:.2f} Å)" for t, d in ss_distances(POSITIONS)) +
        f"; {n_free_sh} free cysteine(s)")
elif n_free_sh:
  print(f"No disulfide bonds: {n_free_sh} cysteine(s) left as free thiols "
        "(SG atoms more than 3 Å apart in the model)")

n_wat = sum(1 for r in TOPOLOGY.residues() if r.name == "HOH")
n_ion = sum(1 for r in TOPOLOGY.residues() if r.name in ("NA", "CL"))
PROTEIN_ATOMS = [a.index for a in TOPOLOGY.atoms() if a.residue.name not in ("HOH", "NA", "CL")]
box = TOPOLOGY.getPeriodicBoxVectors()
print(f"{Force_field}: {TOPOLOGY.getNumAtoms()} atoms = {len(PROTEIN_ATOMS)} protein + "
      f"{n_wat} waters + {n_ion} ions; box edge {box[0][0].value_in_unit(unit.nanometer):.2f} nm")
if TOPOLOGY.getNumAtoms() > 20000:
  print("Note: an extended (unfolded) model needs a large water box, so this run will be slower.")

# protein-only reference structure (the AlphaFold model with hydrogens) for analysis
md.load("design_prepared.pdb").save_pdb("reference.pdb")

In [ ]:
#@title 4. Run the simulation
#@markdown Energy minimization, then a short equilibration with the backbone held near the AlphaFold model, then unrestrained production MD at constant temperature and pressure.
Production_ns = 1 #@param [0.5, 1, 2, 5, 10] {type:"raw"}
Temperature_K = 300 #@param {type:"number"}
Equilibration_ps = 100 #@param {type:"number"}
Frames_per_ns = 100 #@param {type:"number"}

import time
from tqdm.auto import tqdm
from openmm import LangevinMiddleIntegrator, MonteCarloBarostat, CustomExternalForce

TIMESTEP_FS = 4.0
sim_system = openmm.XmlSerializer.deserialize(openmm.XmlSerializer.serialize(system))
# harmonic restraints on CA atoms (switched off for production)
restraint = CustomExternalForce("k_restr*periodicdistance(x,y,z,x0,y0,z0)^2")
restraint.addGlobalParameter("k_restr", 1000.0)  # kJ/mol/nm^2
for p in ["x0", "y0", "z0"]: restraint.addPerParticleParameter(p)
for a in TOPOLOGY.atoms():
  if a.name == "CA" and a.index in PROTEIN_ATOMS:
    restraint.addParticle(a.index, POSITIONS[a.index].value_in_unit(unit.nanometer))
sim_system.addForce(restraint)
sim_system.addForce(MonteCarloBarostat(1.0 * unit.bar, Temperature_K * unit.kelvin))

integrator = LangevinMiddleIntegrator(Temperature_K * unit.kelvin, 1.0 / unit.picosecond,
                                      TIMESTEP_FS * unit.femtoseconds)
platform = openmm.Platform.getPlatformByName(PLATFORM)
props = {"Precision": "mixed"} if PLATFORM in ("CUDA", "OpenCL") else {}
simulation = app.Simulation(TOPOLOGY, sim_system, integrator, platform, props)
simulation.context.setPositions(POSITIONS)

print("Minimizing energy ...")
simulation.minimizeEnergy(maxIterations=2000)
if SS_PAIRS:
  pos = simulation.context.getState(getPositions=True).getPositions()
  print("  disulfides after minimization: " +
        ", ".join(f"{t} {d:.2f} Å" for t, d in ss_distances(pos)) + "  (2.05 Å is typical)")
simulation.context.setVelocitiesToTemperature(Temperature_K * unit.kelvin)

eq_steps = int(Equilibration_ps * 1000 / TIMESTEP_FS)
print(f"Equilibrating {Equilibration_ps} ps with CA restraints ...")
simulation.step(eq_steps)
simulation.context.setParameter("k_restr", 0.0)

prod_steps = int(Production_ns * 1e6 / TIMESTEP_FS)
report = max(1, int(prod_steps / (Production_ns * Frames_per_ns)))
simulation.reporters = [app.DCDReporter("production.dcd", report,
                                        enforcePeriodicBox=False, atomSubset=PROTEIN_ATOMS),
                        app.StateDataReporter("production.log", report, time=True,
                                              potentialEnergy=True, temperature=True,
                                              density=True, speed=True)]
print(f"Production: {Production_ns} ns = {prod_steps} steps of {TIMESTEP_FS:g} fs")
chunk = report * 10
t0 = time.time()
with tqdm(total=prod_steps, unit="step") as bar:
  done = 0
  while done < prod_steps:
    n = min(chunk, prod_steps - done)
    simulation.step(n); done += n; bar.update(n)
wall = time.time() - t0
print(f"Done in {wall/60:.1f} min ({Production_ns / (wall / 86400):.0f} ns/day)")

In [ ]:
#@title 5. Analyze: did the design stay folded?
import matplotlib.pyplot as plt
ref = md.load("reference.pdb")
traj = md.load("production.dcd", top=ref)
ca = traj.topology.select("name CA")
t_ns = np.arange(traj.n_frames) * (Production_ns / traj.n_frames)

traj.superpose(ref, atom_indices=ca)
rmsd = md.rmsd(traj, ref, atom_indices=ca) * 10                  # Å, vs the AlphaFold model
rg = md.compute_rg(traj.atom_slice(ca)) * 10                     # Å
avg = traj.atom_slice(ca)
avg_xyz = avg.xyz.mean(0)
rmsf = np.sqrt(((avg.xyz - avg_xyz) ** 2).sum(-1).mean(0)) * 10  # Å
ss = md.compute_dssp(traj, simplified=True)                      # H helix, E strand, C coil
ss_ref = md.compute_dssp(ref, simplified=True)[0]
helix_t = (ss == "H").mean(1) * 100
helix_res = (ss == "H").mean(0) * 100

fig, ax = plt.subplots(2, 2, figsize=(12, 7.5))
ax[0, 0].plot(t_ns, rmsd, color="k", lw=1)
ax[0, 0].set(xlabel="time (ns)", ylabel="CA RMSD to AlphaFold model (Å)", title="Drift from the design")
ax[0, 1].plot(t_ns, helix_t, color="tab:red", lw=1, label="helix")
ax[0, 1].plot(t_ns, (ss == "E").mean(1) * 100, color="tab:blue", lw=1, label="strand")
ax[0, 1].axhline((ss_ref == "H").mean() * 100, color="tab:red", ls="--", lw=0.8, label="helix in design")
ax[0, 1].set(xlabel="time (ns)", ylabel="% of residues", ylim=(-3, 100), title="Secondary structure content")
ax[0, 1].legend(fontsize=8)
codes = {"H": 0, "E": 1, "C": 2, "NA": 2}
ss_num = np.vectorize(lambda s: codes.get(s, 2))(ss).T
from matplotlib.colors import ListedColormap
ax[1, 0].imshow(ss_num, aspect="auto", cmap=ListedColormap(["tab:red", "tab:blue", "lightgray"]),
                vmin=0, vmax=2, interpolation="nearest",
                extent=[0, Production_ns, len(ca) + 0.5, 0.5])
ax[1, 0].set(xlabel="time (ns)", ylabel="residue", title="DSSP: red helix, blue strand, gray coil")
ax[1, 1].plot(np.arange(1, len(ca) + 1), rmsf, color="k", label="RMSF (MD)")
ax[1, 1].set(xlabel="residue", ylabel="CA RMSF (Å)", title="Flexibility vs AlphaFold confidence")
if len(PLDDT) == len(ca):
  a2 = ax[1, 1].twinx(); a2.plot(np.arange(1, len(ca) + 1), PLDDT, color="tab:orange", label="pLDDT")
  a2.set_ylabel("AlphaFold pLDDT", color="tab:orange"); a2.set_ylim(0, 100)
fig.tight_layout(); fig.savefig("md_analysis.png", dpi=150); plt.show()

half = traj.n_frames // 2
print(f"Sequence: {SEQUENCE}")
print(f"AlphaFold mean pLDDT:              {PLDDT.mean():5.1f}")
print(f"CA RMSD, second half of the run:   {rmsd[half:].mean():5.1f} ± {rmsd[half:].std():.1f} Å")
print(f"Radius of gyration (second half):  {rg[half:].mean():5.1f} Å")
print(f"Helix: design {(ss_ref == 'H').mean()*100:.0f}%  ->  second half of MD {helix_t[half:].mean():.0f}%")

In [ ]:
#@title 6. Watch the trajectory
#@markdown Hydrophobic side chains are drawn with yellow carbons, as in Part 1.
n_show = 50 #@param {type:"slider", min:10, max:100, step:10}
import py3Dmol
step = max(1, traj.n_frames // n_show)
heavy = traj.topology.select("not element H")
movie = traj[::step].atom_slice(heavy)
movie.save_pdb("movie.pdb")
view = py3Dmol.view(width=800, height=480)
view.addModelsAsFrames(open("movie.pdb").read(), "pdb")
HP = ["ALA", "GLY", "VAL", "ILE", "LEU", "PHE", "MET", "PRO", "TRP", "CYS", "TYR"]
view.setStyle({"cartoon": {"color": "spectrum"}})
view.addStyle({"and": [{"resn": HP}, {"atom": ["C", "O", "N"], "invert": True}]},
              {"stick": {"colorscheme": "yellowCarbon", "radius": 0.25}})
view.addStyle({"and": [{"resn": HP, "invert": True}, {"atom": ["C", "O", "N"], "invert": True}]},
              {"stick": {"colorscheme": "whiteCarbon", "radius": 0.25}})
view.zoomTo(); view.animate({"loop": "forward", "interval": 100})
view.show()

In [ ]:
#@title 7. Download the results
import zipfile
out = "md_results.zip"
with zipfile.ZipFile(out, "w") as z:
  for f in ["reference.pdb", "design_prepared.pdb", "production.dcd", "production.log",
            "md_analysis.png", "movie.pdb"]:
    if os.path.isfile(f): z.write(f)
try:
  from google.colab import files
  files.download(out)
except ImportError:
  print("saved", out)